# Exploratory Data Analysis - Predict Future Sales

## 2. Comprensión inicial del dataset

### 2.a Carga de los archivos y primeras filas

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW = Path("..") / "data" / "raw"

sales = pd.read_csv(RAW / "sales_train.csv")
items = pd.read_csv(RAW / "items.csv")
item_categories = pd.read_csv(RAW / "item_categories.csv")
shops = pd.read_csv(RAW / "shops.csv")
test = pd.read_csv(RAW / "test.csv")

# Copia de la fecha original en texto, para validar el formato en 3.b.
fecha_texto = sales["date"].copy()

tablas = {
    "sales_train": sales,
    "items": items,
    "item_categories": item_categories,
    "shops": shops,
    "test": test,
}

In [2]:
sales.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.00,1.0
1,03.01.2013,0,25,2552,899.00,1.0
2,05.01.2013,0,25,2552,899.00,-1.0
3,06.01.2013,0,25,2554,1709.05,1.0
4,15.01.2013,0,25,2555,1099.00,1.0


**Interpretación:** cada fila es una transacción de un producto en una tienda y un día. La fecha viene como texto `dd.mm.aaaa`. Ya en las primeras filas aparece una devolución: el producto 2552 de la tienda 25 se vende el 03.01.2013 y tiene `item_cnt_day = -1` el 05.01.2013.

In [3]:
items.head()

,item_name,item_id,item_category_id
0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,0,40
1,!ABBYY FineReader 12 Professional Edition Full...,1,76
2,***В ЛУЧАХ СЛАВЫ (UNV) D,2,40
3,***ГОЛУБАЯ ВОЛНА (Univ) D,3,40
4,***КОРОБКА (СТЕКЛО) D,4,40


**Interpretación:** es el catálogo que une cada `item_id` con su categoría. Los nombres están en ruso y traen ruido de texto (símbolos como `!` y `***`, y espacios de relleno), por lo que `item_category_id` es más útil que `item_name` para agrupar productos.

In [4]:
item_categories.head()

,item_category_name,item_category_id
0,PC - Гарнитуры/Наушники,0
1,Аксессуары - PS2,1
2,Аксессуары - PS3,2
3,Аксессуары - PS4,3
4,Аксессуары - PSP,4


**Interpretación:** los nombres siguen el patrón `familia - subcategoría` (por ejemplo, `Аксессуары - PS2`, accesorios de PS2), lo que permitiría separar una categoría más general en etapas posteriores.

In [5]:
shops.head()

,shop_name,shop_id
0,"!Якутск Орджоникидзе, 56 фран",0
1,"!Якутск ТЦ ""Центральный"" фран",1
2,"Адыгея ТЦ ""Мега""",2
3,"Балашиха ТРК ""Октябрь-Киномир""",3
4,"Волжский ТЦ ""Волга Молл""",4


**Interpretación:** el nombre de cada tienda empieza con la ciudad (Якутск, Адыгея, Балашиха, Волжский), seguida de datos del centro comercial. Las tiendas 0 y 1 están en Якутск y llevan `!` al inicio, un detalle a revisar cuando se busquen tiendas repetidas en la sección 4.

In [6]:
test.head()

,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233
3,3,5,5232
4,4,5,5268


**Interpretación:** `test` solo contiene los pares tienda-producto a predecir con su `ID`. No trae fecha, precio ni cantidad, así que toda la información para el modelo debe salir del histórico.

### 2.b Tamaño de cada tabla

In [7]:
pd.DataFrame(
    [{"tabla": nombre, "filas": df.shape[0], "columnas": df.shape[1]} for nombre, df in tablas.items()]
).set_index("tabla")

,filas,columnas
tabla,,
sales_train,2935849,6
items,22170,3
item_categories,84,2
shops,60,2
test,214200,3


**Interpretación:** el histórico tiene 2,935,849 transacciones, frente a catálogos pequeños de 22,170 productos, 84 categorías y 60 tiendas. `test` tiene 214,200 filas, que equivalen exactamente a 42 tiendas × 5,100 productos (ver 3.a): se pide predecir la combinación completa de esas tiendas y productos, aunque un par no se haya vendido nunca.

### 2.c Descripción de las columnas y variable objetivo

Descripción de las columnas de las tablas del dataset:

| Columna | Tabla(s) | Tipo | Descripción |
|---|---|---|---|
| `date` | `sales_train` | Fecha (texto `dd.mm.aaaa` en el archivo) | Día en que ocurrió la transacción. |
| `date_block_num` | `sales_train` | Entero ordinal | Número de mes consecutivo: 0 = enero 2013, 33 = octubre 2015. |
| `shop_id` | `sales_train`, `shops`, `test` | Identificador categórico | Identificador único de la tienda (sucursal). |
| `item_id` | `sales_train`, `items`, `test` | Identificador categórico | Identificador único del producto. |
| `item_price` | `sales_train` | Numérica continua | Precio unitario del producto en esa transacción. |
| `item_cnt_day` | `sales_train` | Numérica discreta | Unidades vendidas en el día; los valores negativos representan devoluciones. |
| `item_name` | `items` | Texto | Nombre del producto. |
| `item_category_id` | `items`, `item_categories` | Identificador categórico | Identificador de la categoría del producto. |
| `item_category_name` | `item_categories` | Texto | Nombre de la categoría del producto. |
| `shop_name` | `shops` | Texto | Nombre de la tienda. |
| `ID` | `test` | Identificador | Identificador de cada par tienda-producto a predecir. |

**Variable objetivo:** la suma mensual de `item_cnt_day` por tienda-producto, es decir, las unidades vendidas de cada par (`shop_id`, `item_id`) en cada mes (`date_block_num`). No existe como columna en los archivos: se construye agregando `sales_train`, y debe predecirse para noviembre de 2015 en los pares de `test.csv`.

### 2.d Rango temporal de los datos

In [8]:
fechas = pd.to_datetime(fecha_texto, format="%d.%m.%Y")

pd.Series(
    {
        "fecha mínima": fechas.min().date(),
        "fecha máxima": fechas.max().date(),
        "días en el rango": (fechas.max() - fechas.min()).days + 1,
        "meses calendario distintos": fechas.dt.to_period("M").nunique(),
        "date_block_num mínimo": sales["date_block_num"].min(),
        "date_block_num máximo": sales["date_block_num"].max(),
        "date_block_num distintos": sales["date_block_num"].nunique(),
    },
    name="valor",
).to_frame()

,valor
fecha mínima,2013-01-01
fecha máxima,2015-10-31
días en el rango,1034
meses calendario distintos,34
date_block_num mínimo,0
date_block_num máximo,33
date_block_num distintos,34


**Interpretación:** los datos van del 1 de enero de 2013 al 31 de octubre de 2015, es decir, 1,034 días y 34 meses calendario. `date_block_num` toma exactamente 34 valores (0 a 33), como indica el PDF, y el mes a predecir (noviembre de 2015) es el inmediatamente siguiente al último mes observado.

## 3. Validación de la estructura de los datos

### 3.a Tipos de datos de cada variable

In [9]:
tipo_semantico = {
    "date": "fecha (guardada como texto)",
    "date_block_num": "numérica ordinal (mes)",
    "shop_id": "categórica (identificador)",
    "item_id": "categórica (identificador)",
    "item_price": "numérica continua",
    "item_cnt_day": "numérica discreta",
    "item_name": "categórica (texto)",
    "item_category_id": "categórica (identificador)",
    "item_category_name": "categórica (texto)",
    "shop_name": "categórica (texto)",
    "ID": "categórica (identificador)",
}

pd.DataFrame(
    [
        {
            "tabla": nombre,
            "columna": columna,
            "dtype en pandas": str(df[columna].dtype),
            "tipo semántico": tipo_semantico[columna],
            "valores distintos": df[columna].nunique(),
        }
        for nombre, df in tablas.items()
        for columna in df.columns
    ]
).set_index(["tabla", "columna"])

dtype en pandas               tipo semántico  valores distintos
tabla           columna                                                                           
sales_train     date                           str  fecha (guardada como texto)               1034
                date_block_num               int64       numérica ordinal (mes)                 34
                shop_id                      int64   categórica (identificador)                 60
                item_id                      int64   categórica (identificador)              21807
                item_price                 float64            numérica continua              19993
                item_cnt_day               float64            numérica discreta                198
items           item_name                      str           categórica (texto)              22170
                item_id                      int64   categórica (identificador)              22170
                item_category_id             int64   categórica (identificador)                 84
item_categories item_category_name             str           categórica (texto)                 84
                item_category_id             int64   categórica (identificador)                 84
shops           shop_name                      str           categórica (texto)                 60
                shop_id                      int64   categórica (identificador)                 60
test            ID                           int64   categórica (identificador)             214200
                shop_id                      int64   categórica (identificador)                 42
                item_id                      int64   categórica (identificador)               5100

**Interpretación:** `date` se cargó como texto y debe convertirse a fecha. Los identificadores (`shop_id`, `item_id`, `item_category_id`) son enteros en pandas pero categóricos en significado, así que no tiene sentido promediarlos. En `sales_train` aparecen 21,807 de los 22,170 productos del catálogo, por lo que 363 nunca se vendieron en el histórico. Todas las 60 tiendas tienen ventas, mientras que `test` solo incluye 42 tiendas y 5,100 productos.

In [10]:
# item_cnt_day se carga como decimal: comprobamos si realmente tiene valores fraccionarios.
fraccionarios = (sales["item_cnt_day"] % 1 != 0).sum()
print(f"Valores no enteros en item_cnt_day: {fraccionarios:,}")

Valores no enteros en item_cnt_day: 0


**Interpretación:** `item_cnt_day` está guardada como decimal, pero no tiene ningún valor fraccionario. Es una cantidad entera y puede convertirse a entero sin perder información.

### 3.b Conversión de `date` a datetime

In [11]:
formato_valido = fecha_texto.str.fullmatch(r"\d{2}\.\d{2}\.\d{4}")

sales["date"] = pd.to_datetime(fecha_texto, format="%d.%m.%Y", errors="raise")

pd.Series(
    {
        "filas con formato dd.mm.aaaa": f"{formato_valido.sum():,} de {len(fecha_texto):,}",
        "dtype después de convertir": str(sales["date"].dtype),
        "fechas nulas (NaT)": sales["date"].isna().sum(),
        "fecha mínima": sales["date"].min().date(),
        "fecha máxima": sales["date"].max().date(),
        "archivo ordenado por fecha": sales["date"].is_monotonic_increasing,
    },
    name="resultado",
).to_frame()

,resultado
filas con formato dd.mm.aaaa,"2,935,849 de 2,935,849"
dtype después de convertir,datetime64[us]
fechas nulas (NaT),0
fecha mínima,2013-01-01
fecha máxima,2015-10-31
archivo ordenado por fecha,False


**Interpretación:** las 2,935,849 filas cumplen el formato `dd.mm.aaaa` y la conversión a datetime no generó fechas nulas, con el mismo rango hallado en 2.d. El archivo no está ordenado por fecha, de modo que hay que ordenar o agregar antes de cualquier operación que dependa de la secuencia temporal.

### 3.c Frecuencia temporal: días sin ventas, consistencia de `date_block_num` y frecuencia de agregación

**Días sin ventas a nivel de toda la cadena**

In [12]:
calendario = pd.date_range(sales["date"].min(), sales["date"].max(), freq="D")

diaria = (
    sales.groupby("date")
    .agg(
        unidades_netas=("item_cnt_day", "sum"),
        transacciones=("item_cnt_day", "size"),
        tiendas_activas=("shop_id", "nunique"),
    )
    .reindex(calendario, fill_value=0)
)

pd.Series(
    {
        "días en el calendario": len(calendario),
        "días con al menos un registro": (diaria["transacciones"] > 0).sum(),
        "días sin registros": (diaria["transacciones"] == 0).sum(),
        "días con unidades netas <= 0": (diaria["unidades_netas"] <= 0).sum(),
        "mínimo de tiendas activas en un día": diaria["tiendas_activas"].min(),
        "máximo de tiendas activas en un día": diaria["tiendas_activas"].max(),
    },
    name="valor",
).to_frame()

,valor
días en el calendario,1034
días con al menos un registro,1034
días sin registros,0
días con unidades netas <= 0,0
mínimo de tiendas activas en un día,17
máximo de tiendas activas en un día,52


**Interpretación:** a nivel de cadena no falta ningún día: los 1,034 días del calendario tienen registros y ninguno tiene unidades netas iguales o menores a cero. La serie diaria total está completa y no requiere rellenar fechas. En cambio, el número de tiendas activas por día varía entre 17 y 52 y nunca llega a 60, de modo que la cobertura por tienda no es constante.

In [13]:
diaria.nsmallest(5, "unidades_netas")

,unidades_netas,transacciones,tiendas_activas
2015-05-12,1274.0,1123,44
2015-05-13,1294.0,1128,44
2015-05-14,1302.0,1174,44
2015-10-05,1340.0,1176,41
2015-10-19,1341.0,1201,42


**Interpretación:** incluso los días más bajos registran más de 1,200 unidades netas, así que no hay días anómalamente vacíos. Los cinco mínimos ocurren en 2015, entre mayo y octubre, lo que coincide con la menor actividad de ese año que se observa en la tabla por mes más adelante.

**Días sin ventas por tienda**

In [14]:
inicio, fin = sales["date"].min(), sales["date"].max()

por_tienda = sales.groupby("shop_id").agg(
    primera_venta=("date", "min"),
    ultima_venta=("date", "max"),
    dias_con_venta=("date", "nunique"),
)
por_tienda["dias_en_su_periodo"] = (por_tienda["ultima_venta"] - por_tienda["primera_venta"]).dt.days + 1
por_tienda["dias_sin_venta"] = por_tienda["dias_en_su_periodo"] - por_tienda["dias_con_venta"]
por_tienda["pct_sin_venta"] = (por_tienda["dias_sin_venta"] / por_tienda["dias_en_su_periodo"] * 100).round(1)

pd.Series(
    {
        "tiendas con ventas": len(por_tienda),
        "tiendas activas de principio a fin": ((por_tienda["primera_venta"] == inicio) & (por_tienda["ultima_venta"] == fin)).sum(),
        "tiendas que empiezan después del primer día": (por_tienda["primera_venta"] > inicio).sum(),
        "tiendas que dejan de vender antes del último día": (por_tienda["ultima_venta"] < fin).sum(),
        "tiendas sin ningún hueco en su periodo": (por_tienda["dias_sin_venta"] == 0).sum(),
    },
    name="valor",
).to_frame()

,valor
tiendas con ventas,60
tiendas activas de principio a fin,12
tiendas que empiezan después del primer día,43
tiendas que dejan de vender antes del último día,19
tiendas sin ningún hueco en su periodo,11


**Interpretación:** solo 12 de las 60 tiendas venden desde el primer hasta el último día del histórico. De las otras 48, 43 empiezan después del primer día y 19 terminan antes del último (14 cumplen ambas condiciones). En varios casos la diferencia es mínima, como las tiendas 4, 10 y 24, que empiezan el 2 de enero de 2013, o la tienda 12, cuya última venta es del 30 de octubre de 2015. Solo 11 tiendas no tienen ningún hueco dentro de su propio periodo. Las series por tienda no son homogéneas y un día sin registros puede significar que la tienda aún no operaba, no que vendió cero.

In [15]:
por_tienda.sort_values("dias_sin_venta", ascending=False).head(10)

,primera_venta,ultima_venta,dias_con_venta,dias_en_su_periodo,dias_sin_venta,pct_sin_venta
shop_id,,,,,,
9,2013-10-03,2015-10-14,14,742,728,98.1
20,2014-10-02,2015-10-04,9,368,359,97.6
12,2013-01-03,2015-10-30,826,1031,205,19.9
27,2013-01-01,2015-09-30,899,1003,104,10.4
33,2014-08-29,2015-04-24,201,239,38,15.9
10,2013-01-02,2015-10-31,1001,1033,32,3.1
4,2013-01-02,2015-10-31,1010,1033,23,2.2
24,2013-01-02,2015-10-31,1012,1033,21,2.0
14,2013-01-01,2015-10-31,1014,1034,20,1.9


**Interpretación:** las tiendas 9 (`Выездная Торговля`, venta itinerante) y 20 (`Москва "Распродажа"`, liquidación) venden de forma esporádica: solo 14 y 9 días, con más de 97% de días sin venta dentro de su periodo. Son canales eventuales y conviene tratarlos por separado. La tienda 12 (`Интернет-магазин ЧС`, tienda en línea) tiene 19.9% de días sin venta, y el resto de esta lista está cerca o por debajo de 16%.

**Consistencia de `date_block_num` con la fecha**

In [16]:
bloque_esperado = (sales["date"].dt.year - 2013) * 12 + sales["date"].dt.month - 1
print(f"Filas donde date_block_num no coincide con la fecha: {(sales['date_block_num'] != bloque_esperado).sum():,}")

periodo = sales["date"].dt.to_period("M")
por_bloque = sales.groupby("date_block_num").agg(
    primera_fecha=("date", "min"),
    ultima_fecha=("date", "max"),
    dias_con_registros=("date", "nunique"),
    transacciones=("date", "size"),
)
por_bloque["meses_distintos"] = periodo.groupby(sales["date_block_num"]).nunique()
por_bloque["dias_del_mes"] = por_bloque["primera_fecha"].dt.days_in_month
por_bloque

Filas donde date_block_num no coincide con la fecha: 0


,primera_fecha,ultima_fecha,dias_con_registros,transacciones,meses_distintos,dias_del_mes
date_block_num,,,,,,
0,2013-01-01,2013-01-31,31,115690,1,31
1,2013-02-01,2013-02-28,28,108613,1,28
2,2013-03-01,2013-03-31,31,121347,1,31
3,2013-04-01,2013-04-30,30,94109,1,30
4,2013-05-01,2013-05-31,31,91759,1,31
5,2013-06-01,2013-06-30,30,100403,1,30
6,2013-07-01,2013-07-31,31,100548,1,31
7,2013-08-01,2013-08-31,31,104772,1,31
8,2013-09-01,2013-09-30,30,96137,1,30


**Interpretación:** `date_block_num` es totalmente consistente con la fecha: ninguna fila difiere del valor calculado, cada bloque corresponde a un único mes y cada mes tiene registros en todos sus días. Se puede usar con seguridad como índice mensual. Las transacciones suben con fuerza en diciembre (143,246 en 2013 y 130,786 en 2014) y caen entre años: octubre pasa de 94,202 en 2013 a 79,361 en 2014 y 53,514 en 2015.

**Comparación de frecuencias de agregación**

In [17]:
semana = sales["date"].dt.to_period("W")
dias_por_semana = sales.groupby(semana)["date"].nunique()
semanas_completas = dias_por_semana[dias_por_semana == 7].index

n_pares = sales[["shop_id", "item_id"]].drop_duplicates().shape[0]
niveles = {"diaria": sales["date"], "semanal": semana, "mensual": sales["date_block_num"]}

filas = []
for nombre, clave in niveles.items():
    serie = sales.groupby(clave)["item_cnt_day"].sum()
    if nombre == "semanal":
        serie = serie[serie.index.isin(semanas_completas)]
    observadas = pd.DataFrame(
        {"periodo": clave, "shop_id": sales["shop_id"], "item_id": sales["item_id"]}
    ).drop_duplicates().shape[0]
    posibles = clave.nunique() * n_pares
    filas.append(
        {
            "frecuencia": nombre,
            "periodos": clave.nunique(),
            "combinaciones periodo-tienda-producto posibles": posibles,
            "combinaciones con registros": observadas,
            "% con registros": round(observadas / posibles * 100, 2),
            "CV de las ventas totales": round(serie.std() / serie.mean(), 3),
        }
    )

print(f"Pares tienda-producto con historial: {n_pares:,}")
cruzan_mes = (semana.dt.start_time.dt.to_period("M") != semana.dt.end_time.dt.to_period("M"))
print(f"Semanas que cruzan dos meses: {semana[cruzan_mes].nunique()} de {semana.nunique()}")
pd.DataFrame(filas).set_index("frecuencia")

Pares tienda-producto con historial: 424,124


Semanas que cruzan dos meses: 30 de 148


,periodos,combinaciones periodo-tienda-producto posibles,combinaciones con registros,% con registros,CV de las ventas totales
frecuencia,,,,,
diaria,1034,438544216,2935821,0.67,0.449
semanal,148,62770352,2317130,3.69,0.317
mensual,34,14420216,1609124,11.16,0.265


**Interpretación y conclusión:** conviene agregar a frecuencia **mensual**, por cuatro razones basadas en estos resultados:

1. La variable objetivo y el periodo a predecir son mensuales, y `date_block_num` ofrece un índice mensual sin errores.
2. Por combinación periodo-tienda-producto, solo 0.67% tiene registros a nivel diario y 3.69% a nivel semanal, frente a 11.16% a nivel mensual.
3. La variabilidad relativa de las ventas totales baja al agregar: el coeficiente de variación pasa de 0.449 (diaria) a 0.317 (semanal) y 0.265 (mensual).
4. La semana no encaja con el objetivo: 30 de las 148 semanas cruzan dos meses.

La limitación es que solo hay 34 observaciones mensuales, menos de tres ciclos anuales completos. Por eso la serie total también debe analizarse a nivel diario, como recomienda el PDF para la sección 9, y para estudiar los ciclos por día de la semana. Aun a nivel mensual, cerca de 89% de las combinaciones no tiene registros, lo que habrá que tener en cuenta al preparar el modelo.